# ⏳ The Foundational Leap: Deterministic Neural ODEs
Before the integration of explicit physics equations (PINNs) or the shift to sequence-based Transformers (PINNsFormer), the **Neural Ordinary Differential Equation (NODE)** was the original breakthrough for continuous-time modeling in scientific machine learning. 

This notebook steps back to explore the theory behind this foundational architecture, demonstrating how deep learning transitioned from discrete layers to continuous depth.

---

## 1. The Core Concept: Continuous Depth (Time)
Standard deep neural networks, specifically Residual Networks (ResNets), transform hidden states through a sequence of discrete layers:
$$h_{t+1} = h_t + f(h_t, \theta)$$
*(Where $h_t$ is the hidden state at layer $t$, and $f$ is a neural network layer parameterized by weights $\theta$.)*

In 2018, researchers realized that this equation is mathematically identical to the **Euler method** for solving an ordinary differential equation with a step size of 1. 

If you take the limit as the step size approaches zero, the layers merge into a continuous continuum. Instead of a neural network mapping discrete steps, the neural network becomes the derivative itself:
$$\frac{dh(t)}{dt} = f(h(t), t, \theta)$$

In a Neural ODE, $f$ is your Multi-Layer Perceptron (MLP). It takes the current state $h(t)$ and the continuous time $t$ as inputs, and outputs the instantaneous rate of change (the gradient).

## 2. The Forward Pass: The Black-Box ODE Solver
In a standard neural network, a forward pass involves multiplying matrices layer by discrete layer. 

In a Neural ODE, the forward pass involves handing your neural network over to a dedicated, highly optimized mathematical **ODE Solver** (like Runge-Kutta, Dormand-Prince, or Tsit5).

To find the state of the system at any future time $t_1$, the solver evaluates the integral of your neural network:
$$h(t_1) = h(t_0) + \int_{t_0}^{t_1} f(h(t), t, \theta) dt$$

The solver dynamically chooses how many steps it needs to take to maintain mathematical accuracy. This is highly efficient for smooth systems, but it can bottleneck heavily if the underlying dynamics become stiff (which requires explicit handling, as seen in the PI-NODE framework).

## 2. The Forward Pass: The Black-Box ODE Solver
In a standard neural network, a forward pass involves multiplying matrices layer by discrete layer. 

In a Neural ODE, the forward pass involves handing your neural network over to a dedicated, highly optimized mathematical **ODE Solver** (like Runge-Kutta, Dormand-Prince, or Tsit5).

To find the state of the system at any future time $t_1$, the solver evaluates the integral of your neural network:
$$h(t_1) = h(t_0) + \int_{t_0}^{t_1} f(h(t), t, \theta) dt$$

The solver dynamically chooses how many steps it needs to take to maintain mathematical accuracy. This is highly efficient for smooth systems, but it can bottleneck heavily if the underlying dynamics become stiff (which requires explicit handling, as seen in the PI-NODE framework).

## 3. The Mathematics: Deterministic Optimization
The goal of the deterministic NODE model is to find the single, absolute best set of weights $\theta^*$ that maps out a continuous trajectory perfectly matching your observed data.

Your loss function simply compares the trajectory predicted by the ODE solver against your true empirical data points:
$$\mathcal{L}_{Data}(\theta) = \sum_{i} \| \text{ODESolve}(h(t_0), f, t_i, \theta) - \text{Data}_i \|^2$$

We then minimize this objective using standard continuous gradient descent: $\min_{\theta} \mathcal{L}_{Data}(\theta)$.

## 4. The Engine: The Adjoint Sensitivity Method
This presents a massive technical hurdle: **How do you backpropagate through an entire numerical ODE solver?**

If you use standard AutoDiff to track and store every single internal micro-step the ODE solver takes, your computer will quickly run out of memory. 

Neural ODEs solved this using the **Adjoint Sensitivity Method**. Instead of remembering the forward pass, the Adjoint Method mathematically defines a second ODE that runs *backwards* in time.

Let $a(t) = \frac{\partial \mathcal{L}}{\partial h(t)}$ be the "adjoint" state. The backwards ODE is defined as:
$$\frac{da(t)}{dt} = -a(t)^T \frac{\partial f}{\partial h}$$

By solving this adjoint ODE backwards from the final time block to the start, the network computes the exact gradients of the loss with respect to the weights $\left( \frac{\partial \mathcal{L}}{\partial \theta} \right)$ with an **$\mathcal{O}(1)$ memory cost**. It is incredibly efficient.

## 5. The Deterministic Flaw: The Single Fixed Curve
While mathematically elegant, the deterministic Neural ODE has a primary limitation: **It outputs one fixed curve.**

Because the optimizer just finds the best point-estimate for $\theta^*$, the Neural ODE assumes its prediction is flawlessly accurate. If you train it on sparse data (e.g., sensor measurements taken only every 10 ms), the network will confidently draw a perfectly smooth line connecting those points. 

**It cannot quantify its trajectory uncertainty (epistemic error bars) in the gaps where no data exists.** 

This framework is excellent for discovering unknown smooth dynamics when you have clean, continuous data, but it lacks the strict "physics constraints" of a PINN and the "uncertainty awareness" of a Bayesian model.

In [ ]:
# %% Cell 1: Setup and Data
using Lux, DifferentialEquations, SciMLSensitivity, Optimization, OptimizationOptimisers, Statistics, Random, ComponentArrays, Zygote, Plots

# Assuming t_train (shape: N) and z_train (shape: 4 x N) are already defined.
# t_train = Float32.(df_ordered.timestamp)
# z_train = Float32.(Matrix(df_ordered[:, [:V, :n, :m, :h]])')

rng = Random.default_rng()
Random.seed!(rng, 42)

In [ ]:
# %% Cell 2: Continuous-Time MLP
# The Neural Network that will act as the ODE derivative function: du/dt = NN(u)
nn_ode = Lux.Chain(
    Lux.Dense(4 => 32, tanh),
    Lux.Dense(32 => 32, tanh),
    Lux.Dense(32 => 4) 
)

# Initialize network parameters and states on the CPU
ps, st = Lux.setup(rng, nn_ode)
p_initial = ComponentArray(ps)

In [ ]:
# %% Cell 3: Neural ODE Definition

function neural_dynamics(u, p, t)
    # The neural network takes the current state `u` and predicts the gradient `du/dt`
    # Lux returns a tuple (output, state), we only want the output [1]
    return nn_ode(u, p, st)[1]
end

# Extract the initial condition from the first data point
u0 = z_train[:, 1]
tspan = (t_train[1], t_train[end])

# Define the continuous-time ODE Problem
node_prob = ODEProblem(neural_dynamics, u0, tspan, p_initial)

In [ ]:
# %% Cell 4: Solvers and Data Loss

function predict_trajectory(θ)
    # Remake the ODE problem with the updated neural network weights
    _prob = remake(node_prob, p=θ)
    
    # Solve the ODE. 
    # Tsit5() is a standard fast solver. If the data is highly stiff (like HH action potentials), 
    # AutoTsit5(Rosenbrock23()) might be required to prevent solver stalls.
    # InterpolatingAdjoint forces O(1) memory backpropagation.
    sol = solve(_prob, Tsit5(), 
                saveat=t_train, 
                abstol=1e-6, reltol=1e-6,
                sensealg=InterpolatingAdjoint(autojacvec=ZygoteVJP()))
                
    return sol
end

function data_loss(θ, _)
    sol = predict_trajectory(θ)
    
    # If the ODE solver fails (becomes unstable), return infinite loss
    if sol.retcode != ReturnCode.Success
        return Inf 
    end
    
    pred_data = Array(sol)
    
    # Pure Data Loss: No physics residuals here!
    mse_loss = mean(abs2, pred_data .- z_train)
    
    return mse_loss
end

In [ ]:
# %% Cell 5: Training the Pure Neural ODE

loss_history = Float32[]

callback_fn = function (θ, loss_val)
    push!(loss_history, loss_val)
    if length(loss_history) % 20 == 0
        println("Epoch $(length(loss_history)) | Pure Data Loss: $(round(loss_val, digits=5))")
    end
    return false
end

# Setup AutoZygote Optimization
optf = OptimizationFunction(data_loss, Optimization.AutoZygote())
optprob = OptimizationProblem(optf, p_initial)

println("Starting Pure Neural ODE Training...")
# Train the network using Adam
result = solve(optprob, Adam(0.005), maxiters = 1000, callback = callback_fn)
println("Training Complete!")

# Extract final optimized parameters
p_opt = result.u

# Evaluate the final learned trajectory
final_sol = predict_trajectory(p_opt)
pred_array = Array(final_sol)

# Plotting the result for Voltage
plot(t_train, pred_array[1, :], 
    label="Neural ODE Prediction", 
    linewidth=2, 
    color=:cyan,
    title="Pure Neural ODE (No Physics)",
    xlabel="Time (ms)",
    ylabel="Voltage (mV)"
)
scatter!(t_train, z_train[1, :], 
    label="True Data", 
    markersize=3, 
    color=:orange,
    alpha=0.6
)